# 第 14 章　比較条件と結論

年間貸出件数が約 65% 増えたと聞いて，夏の一日も同じ割合で増えたと考えてよいでしょうか。年間の合計だけでは，季節ごとの利用や，特に貸出が多い日が何日あったかは分かりません。

講義では，一年全体から月別，夏と冬へと対象を替え，平均・中央値・基準以上の日数を比べます。実習では別の季節を選び，その時期に備える担当者へ，何の件数がどれくらい変わったかを報告します。

前半は，章本文の説明に合わせて，上から順にセルを実行します。コードの前に何を調べるかを読み，実行後は表や図のどの値が問いへの答えになるかを確かめてください。後半の約 20 分の実習では，比べる対象や条件を自分で選び，結果を短い文章にします。前半のコードは実行できる見本なので，作図の処理をすべて書き写す必要はありません。

Colab では「ドライブにコピーを保存」で自分用のノートを作ります。CSV は，手元にファイルがなければ公開 URL から読み込みます。ローカルでは，配布 ZIP の配置のまま開くか，ノートと CSV を同じフォルダーに置いて実行できます。

In [ ]:
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import pandas as pd

file_name = "bike-sharing-daily.csv"
csv_path = Path("data") / file_name
if not csv_path.is_file():
    csv_path = Path("../data") / file_name
if not csv_path.is_file():
    csv_path = Path(file_name)
if not csv_path.is_file():
    csv_path = "https://raw.githubusercontent.com/sohei-takagi/python-ds-course-materials/main/data/" + file_name

daily = pd.read_csv(csv_path, parse_dates=["dteday"])
daily["year"] = daily["dteday"].dt.year
daily["month"] = daily["dteday"].dt.month
print(f"{len(daily)} rows, {daily['dteday'].nunique()} distinct dates")
daily[["dteday", "year", "month", "cnt"]].head(3)

## 本文と一緒に実行

### 1. 年間合計と一日平均を区別する

約 65% 増えたという報告は，一年全体の貸出件数の話でしょうか，一日当たりの件数の話でしょうか。年ごとの日数・合計・日平均を同じ表に並べ，それぞれの増加率を求めます。

増加率の計算では，2012 年の値から 2011 年の値を引き，その差を 2011 年の値で割ります。合計と日平均で，何を分母にしたかを見ながら実行してください。

In [ ]:
def year_summary(records):
    return records.groupby("year")["cnt"].agg(
        days="size",
        total="sum",
        daily_mean="mean",
        daily_median="median",
    )


annual = year_summary(daily)
annual.round(2)

In [ ]:
year_values = annual[["total", "daily_mean"]]
extra = year_values.loc[2012] - year_values.loc[2011]
percent_more = extra / year_values.loc[2011] * 100
pd.DataFrame({"difference": extra, "increase_pct": percent_more}).round(2)

年間合計は 64.88%，一日平均は 64.43% 増えました。両者が少し違う理由を，2011 年は 365 日，2012 年は 366 日だったことから説明してください。

ここまでで一年全体の増加は分かりました。しかし，夏だけでも同じ増え方だったかは分かりません。次に，平均のもとになった各日の件数へ戻ります。

### 2. 一年の中で多い時期と少ない日を調べる

どの時期にも年平均と同じくらいの貸出があったのでしょうか。日付順の折れ線を描き，その年の平均より多い日が続く時期と，少ない日が続く時期を探します。

2012 年 10 月末の低い点に注目し，前後の日の件数を表で確かめます。年間の実績を伝えるときと，通常どおり営業した日の利用を知りたいときで，同じ日を集計に含めてよいかを考えます。

In [ ]:
plt.rcParams["figure.figsize"] = (7, 3.6)
fig, ax = plt.subplots(layout="constrained")
for year, color in [(2011, "#607a88"), (2012, "#24686a")]:
    records = daily.loc[daily["year"] == year].sort_values("dteday")
    ax.plot(records["dteday"], records["cnt"], color=color, linewidth=0.9)
    ax.hlines(
        annual.loc[year, "daily_mean"], records["dteday"].min(), records["dteday"].max(),
        color="#313b45", linestyles="dashed", linewidth=1.35,
    )

sandy_day = daily.loc[daily["dteday"] == pd.Timestamp("2012-10-29")].iloc[0]
ax.scatter(sandy_day["dteday"], sandy_day["cnt"], color="#a34f30",
           edgecolors="white", s=45, zorder=3, clip_on=False)
ax.annotate("Oct 29", (sandy_day["dteday"], sandy_day["cnt"]),
            xytext=(-5, 38), textcoords="offset points", ha="right",
            arrowprops={"arrowstyle": "-", "color": "#a34f30"})
ax.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1, 4, 7, 10]))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
ax.tick_params(axis="x", labelrotation=32)
ax.set_xlim(daily["dteday"].min() - pd.Timedelta(days=7), daily["dteday"].max())
ax.set(xlabel="Date", ylabel="Rentals per day", ylim=(0, 9200))
plt.show()

In [ ]:
end_of_october = daily["dteday"].between("2012-10-27", "2012-10-31")
daily.loc[end_of_october, ["dteday", "cnt"]]

10 月 29 日の 22 件も，この集計版に記録された貸出の実績です。年間の貸出合計には含めます。一方，通常どおり営業した日の利用だけを調べるなら，件数が小さいという理由だけで日を外すのではなく，営業時間や休止の記録が必要です。

本文では，第 11 章で確認した時間別記録にも触れています。日別の低い点を見つけたら，その数が何時間分の記録からできているかも確かめます。

### 3. 前年の同じ月と比べる

一年の中には，利用の多い時期と少ない時期がありました。各月でも前年より一律に約 65% 増えたのでしょうか。2011 年と 2012 年の同じ月の日平均を並べ，月ごとの増加率を求めます。

上の図では，同じ月の二つの平均を比べます。下の図では，その差を 2011 年の平均で割った増加率を読みます。どの月でも増えたかと，どの月でも同じ割合で増えたかを分けて確かめてください。

In [ ]:
month_values = daily.groupby(["month", "year"])["cnt"].mean()
monthly = month_values.unstack("year")
monthly["increase_pct"] = (monthly[2012] - monthly[2011]) / monthly[2011] * 100
monthly.loc[[3, 6, 9, 12]].round(2)

In [ ]:
plt.rcParams["figure.figsize"] = (7, 5.1)
fig, (levels, rates) = plt.subplots(2, 1, sharex=True, layout="constrained")
levels.plot(monthly.index, monthly[2011], color="#607a88", marker="o",
            linestyle="--", label="2011")
levels.plot(monthly.index, monthly[2012], color="#24686a", marker="s", label="2012")
levels.set(ylabel="Mean rentals per day", ylim=(0, 8200))
levels.legend(frameon=False, ncols=2)
rates.bar(monthly.index, monthly["increase_pct"], color="#a16142", width=0.68)
rates.axhline(percent_more["daily_mean"], color="#3c4854", linestyle="--",
             linewidth=1.2)
rates.set(xlabel="Calendar month", ylabel="Increase (%)", xticks=range(1, 13),
          xlim=(0.45, 12.55), ylim=(0, 175))
plt.show()

12 か月すべてで，2012 年の日平均が前年の同じ月を上回りました。ただし，増加率は約 41〜157% と月によって違います。

「どの月でも増えた」とは言えても，「どの月も約 65% 増えた」とは言えません。夏の利用を説明するために，次は夏に当たる月だけをまとめて比べます。

### 4. 夏と冬の増加率と件数を比べる

前年からの増加率が大きい季節ほど，実際の貸出件数も多いのでしょうか。夏は 6〜8 月，冬は各暦年の 1・2・12 月として，両年の同じ月を比べます。ここでの冬は，年をまたぐ連続した三か月ではありません。

一年全体と同じ関数を使って集計し，日平均の増加件数と増加率を求めます。図では，二つの年の平均を結ぶ線の長さと，2012 年を示す右端の位置を区別して読みます。

In [ ]:
summer = daily.loc[daily["month"].isin([6, 7, 8])]
winter = daily.loc[daily["month"].isin([1, 2, 12])]

summer_summary = year_summary(summer)
winter_summary = year_summary(winter)
seasonal = pd.concat(
    [summer_summary, winter_summary],
    keys=["Jun–Aug", "Jan, Feb, Dec"],
    names=["period"],
)
seasonal.round(1)

In [ ]:
for period, summary in [("Jun–Aug", summer_summary), ("Jan, Feb, Dec", winter_summary)]:
    previous_mean = summary.loc[2011, "daily_mean"]
    increase = summary.loc[2012, "daily_mean"] - previous_mean
    increase_pct = increase / previous_mean * 100
    print(f"{period}: +{increase:,.0f} per day, +{increase_pct:.1f}%")

In [ ]:
plt.rcParams["figure.figsize"] = (6.5, 2.8)
fig, ax = plt.subplots(layout="constrained")
for row, summary in [(1, summer_summary), (0, winter_summary)]:
    first = summary.loc[2011, "daily_mean"]
    second = summary.loc[2012, "daily_mean"]
    ax.hlines(row, first, second, color="#8b98a3", linewidth=2.4)
    ax.scatter(first, row, color="#607a88", s=50, marker="o", zorder=3)
    ax.scatter(second, row, color="#24686a", s=50, marker="s", zorder=3)
    ax.annotate(f"{first:,.0f}", (first, row), xytext=(0, 9),
                textcoords="offset points", ha="center")
    ax.annotate(f"{second:,.0f}", (second, row), xytext=(0, 9),
                textcoords="offset points", ha="center")
ax.scatter([], [], color="#607a88", s=45, marker="o", label="2011")
ax.scatter([], [], color="#24686a", s=45, marker="s", label="2012")
ax.set(xlabel="Mean rentals per day", xlim=(0, 7600), ylim=(-0.45, 1.55),
       yticks=[0, 1], yticklabels=["Jan, Feb, Dec", "Jun–Aug"])
ax.legend(loc="upper left", frameon=False, ncols=2)
plt.show()

日平均の増加率は夏が約 47.3%，冬が約 84.2% です。一方，増えた件数と 2012 年の平均件数は，どちらも夏の方が大きくなります。前年の平均を分母にする増加率と，件数そのものは別の数です。

図の線の長さから何件増えたかを，右端の位置から 2012 年に一日平均何件あったかを説明してください。利用が多い日に備えるには，さらに日ごとの分布も調べる必要があります。

### 5. 平均・中央値・基準以上の日数を使い分ける

一日を代表する件数として，平均と中央値は同じ情報を表しているでしょうか。まず三日分の小さな例で，低い一日の値を替えたとき，二つの代表値がどう変わるかを確かめます。

次に，2012 年の夏と冬で，一日 6,500 件以上の貸出があった日を数えます。平均や中央値だけでは，この基準に達した日数は分かりません。日別件数の分布を描き，元の各日の値が基準以上かを判定します。

In [ ]:
three_counts = pd.Series([22, 5566, 7852])
replaced_counts = pd.Series([1096, 5566, 7852])
display(pd.DataFrame({"Original": three_counts.agg(["mean", "median"]),
                      "One day replaced": replaced_counts.agg(["mean", "median"])}))
annual[["days", "daily_mean", "daily_median"]]

In [ ]:
plt.rcParams["figure.figsize"] = (6.8, 3.8)
busy_threshold = 6500
summer_2012 = summer.loc[summer["year"] == 2012, "cnt"]
winter_2012 = winter.loc[winter["year"] == 2012, "cnt"]
fig, axes = plt.subplots(1, 2, sharex=True, sharey=True, layout="constrained")
for ax, counts, title, color in [
    (axes[0], summer_2012, "Jun–Aug (92 days)", "#24686a"),
    (axes[1], winter_2012, "Jan, Feb, Dec (91 days)", "#7b8293"),
]:
    ax.hist(counts, bins=range(0, 9001, 500), color=color, edgecolor="white")
    ax.axvline(busy_threshold, linestyle="--", color="#a34f30", linewidth=1.6)
    ax.set(title=title, xlabel="Rentals per day", xlim=(0, 9000),
           xticks=[0, 3000, 6500, 9000])
axes[0].set_ylabel("Days")
plt.show()

In [ ]:
for period, counts in [("Jun–Aug", summer_2012), ("Jan, Feb, Dec", winter_2012)]:
    busy_days = (counts >= busy_threshold).sum()
    print(f"{period}: {busy_days} / {len(counts)} days")

小例では，一つの低い値を替えると合計が変わり，平均も変わります。一方，並べたときの真ん中の値が変わらなければ，中央値は同じです。何を代表させたいかによって，見る数が違います。

6,500 件以上の日数は，日ごとの値を直接判定して数えた結果です。夏と冬の該当日数と分母を読み，増加率の大小とは違う説明になることを確かめてください。6,500 件は教材で置いた目安であり，実際の設備能力や必要な自転車台数を表してはいません。

## Jupyter 実習（約 20 分）

夏についての報告を，別の季節に備える人向けに書き直します。3〜5 月か 9〜11 月を選び，同じ関数で両年の利用を比べてください。

調べたいのは，選んだ季節でも前年より増えたかと，増えた件数・割合が夏とどう違うかです。結果を見る前に対象を決め，夏の結論をそのまま使わず，自分が選んだ期間の値で説明します。

## 1. 調べる時期を決める（約 5 分）

3〜5 月か 9〜11 月のどちらかを選びます。例えば「秋の貸出に備える担当者が，秋の一日について知りたい」のように，選ぶ理由を一文で書きます。選んだ時期の日平均の増加率が，6〜8 月と同じくらいになりそうかを予想し，理由を添えます。

次のセルには本文で使った関数が入っています。セルを実行すると，6〜8 月の日数，期間内の合計，一日平均，日別件数の中央値を年ごとに表示します。

（ここに選んだ時期，理由，予想を記入）

In [ ]:
def year_summary(records):
    return records.groupby("year")["cnt"].agg(
        days="size",
        total="sum",
        daily_mean="mean",
        daily_median="median",
    )


summer = daily.loc[daily["month"].isin([6, 7, 8])]
summer_summary = year_summary(summer)
summer_summary.round(2)

## 2. 選んだ月を同じ方法で集計する（約 9 分）

課題セルでは，`months` に選んだ三つの月をリストで入れます。次に，`month` がそのリストに含まれる日だけを抽出し，`chosen` に入れます。実行後に，両年の日付の範囲と日数を確認し，選んだ季節の記録が入っているかを確かめてください。

続くセルは，`chosen` を講義と同じ関数へ渡し，6〜8 月との比較表を作ります。各年の平均と中央値を読み，どちらにも何日分の記録を使ったかを記録します。

平均と中央値の増加率は，それぞれ 2011 年の値を分母に計算します。増加率だけでなく，2012 年の一日平均が何件だったかも見て，その季節にどれくらいの利用があったかを説明できるようにします。

In [ ]:
months = None
# ここに，選んだ三つの月を months に入れる一行を書く。

chosen = None
if months is not None:
    # ここに，月の条件で日を選び chosen に入れる一行を書く。
    pass

if chosen is None:
    print("月のリストと抽出の両方を書くと，選んだ日数が出ます。")
else:
    check = chosen.groupby("year")["dteday"].agg(days="size", first="min", last="max")
    display(check)

In [ ]:
if chosen is None:
    print("日付の選択まで書いてから，このセルを実行します。")
else:
    chosen_summary = year_summary(chosen)
    comparison = pd.concat(
        [summer_summary, chosen_summary],
        keys=["Jun–Aug", str(months)],
        names=["period"],
    )
    display(comparison[["days", "total", "daily_mean", "daily_median"]].round(2))

    changes = []
    for period, summary in [("Jun–Aug", summer_summary), (str(months), chosen_summary)]:
        daily_values = summary[["daily_mean", "daily_median"]]
        first_year = daily_values.loc[2011]
        difference = daily_values.loc[2012] - first_year
        growth = difference / first_year * 100
        changes.append(growth.rename(period))

    change_table = pd.DataFrame(changes)
    change_table.columns = ["Mean increase (%)", "Median increase (%)"]
    display(change_table.round(2))

<details>
<summary>月の入力と選択の書き方</summary>

まず講義の夏と冬の抽出で，月のリストと `.isin()` がどのように使われたかを読み返してください。書き方を確かめたい場合は，次の例を使います。3〜5 月を選んだ場合は，`months = None` の下に次の一行を書きます。9〜11 月を選んだ場合はリストの三つの値を変えます。

```python
months = [3, 4, 5]
```

`if months is not None:` の中で，`month` がそのリストにある日を選びます。

```python
    chosen = daily.loc[daily["month"].isin(months)]
```

</details>

## 3. 選んだ季節について報告する（約 6 分）

選んだ時期の貸出体制を考える担当者に向けて，次の三点が伝わるよう 3〜4 文で報告します。

1. 比べた時期と両年の日数は何か。一日平均と中央値はそれぞれ何件から何件になったか。
2. 選んだ時期でも 2012 年の方が多かったか。6〜8 月と比べて日平均の増加率がどう違い，2012 年の平均件数そのものはどう違ったか。
3. 選んだ時期の件数を翌年にも当てはめたいなら，何を追加で調べるか。

（ここに報告を記入）

カーネルを再起動し，上から全セルを実行します。比較表と文章を残したノートブックを提出します。

### 一日の目安を替える（任意）

選んだ時期の 2012 年に限って，6,500 件以上の日と 7,000 件以上の日をそれぞれ数えます。二つとも練習用の仮の目安です。対象の日付を変えずに目安だけを高くしたとき，該当する日が何日減るかを調べます。平均件数が分かっても，この日数までは決まらない理由を，日別の値の分布から考えてください。

### 出典

Capital Bikeshare の日別集計は，Hadi Fanaee-T が UCI Machine Learning Repository で公開した `day.csv` を教材用に改名したものです。 [データ配布元](https://doi.org/10.24432/C5W894) ・ [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/)